# Extraction de taux de change via l'API OANDA Exchange Rates

Ce notebook se connecte à l'API (`https://api.corporatefxservices.com`), puis collecte des taux spot, un historique journalier, des taux forward, et exporte les résultats.

Source : spécification OpenAPI officielle, version 2.0.0 (copie dans `docs/reference/`).

Avant de commencer :
- Une clé API est nécessaire (essai gratuit de 7 jours sur le site OANDA FX Data Services).
- Elle est envoyée dans l'en-tête `Authorization: Bearer <clé>`.
- Chaque devise de cotation renvoyée compte comme un quote sur votre abonnement.
- Les taux forward, l'open/close et certaines données dépendent de votre abonnement.

In [ ]:
# Dépendances (à lancer une fois si besoin) : requests, pandas, matplotlib, openpyxl
# %pip install requests pandas matplotlib openpyxl

## 1. Connexion

In [ ]:
import os, time, getpass
import requests
import pandas as pd

BASE_URL = "https://api.corporatefxservices.com"

# La clé est lue dans la variable d'environnement OANDA_FX_API_KEY, sinon demandée à la saisie.
# Ne l'écrivez jamais en dur dans le notebook.
API_KEY = os.environ.get("OANDA_FX_API_KEY") or getpass.getpass("Clé API OANDA : ")

session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"


def api_get(path, retries=3, **params):
    """Appelle un endpoint (sans extension) et renvoie le JSON décodé.

    Réessaie avec attente croissante sur erreur réseau, 429 et 5xx.
    Lève une erreur explicite sur les autres codes (400, 401, 403, 404...).
    """
    params = {k: v for k, v in params.items() if v is not None}
    for attempt in range(retries):
        try:
            r = session.get(f"{BASE_URL}{path}.json", params=params, timeout=30)
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == retries - 1:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        if not r.ok:
            raise RuntimeError(f"HTTP {r.status_code} sur {path} : {r.text[:300]}")
        restants = r.headers.get("X-Rate-Limit-Remaining")
        if restants is not None:
            print("Quotes restants :", restants)
        return r.json()

## 2. Test de la connexion

In [ ]:
print(api_get("/v2/remaining_quotes"))

In [ ]:
devises = pd.DataFrame(api_get("/v2/currencies")["currencies"])
print(len(devises), "devises")
devises.head()

## 3. Taux spot (Bid, Ask, Mid)

`base` et `quote` acceptent plusieurs devises. Sans `date_time`, le taux temps réel est renvoyé (mise à jour toutes les 5 secondes). Avec `date_time`, il faut un instant passé sur un pas de 15 minutes, par exemple `2025-06-02T10:15:00+00:00` (accès selon abonnement).

In [ ]:
def spot(base, quote=None, date_time=None, data_set=None):
    data = api_get("/v2/rates/spot", base=base, quote=quote, date_time=date_time, data_set=data_set)
    df = pd.DataFrame(data["quotes"])
    for c in ("bid", "ask", "midpoint"):
        df[c] = pd.to_numeric(df[c])
    df["date_time"] = pd.to_datetime(df["date_time"], utc=True)
    return df


taux_spot = spot(base=["EUR"], quote=["USD", "GBP", "CHF"])
taux_spot

## 4. Historique journalier

L'endpoint `/v2/rates/candles` renvoie un chandelier par jour (open, close, average, high, low, en bid, ask et midpoint selon `fields`). Une requête est limitée à 10 000 (jours x paires), donc la plage est découpée automatiquement. L'open et le close peuvent exiger un abonnement supérieur.

In [ ]:
def historique(base, quote, debut, fin, fields=("averages", "highs", "lows"), data_set=None):
    base = [base] if isinstance(base, str) else list(base)
    quote = [quote] if isinstance(quote, str) else list(quote)
    paires = len(base) * len(quote)
    jours_max = max(1, min(10_000 // paires, 365))   # limite documentée : 10 000 (jours x paires)

    morceaux = []
    cur = pd.Timestamp(debut)
    fin = pd.Timestamp(fin)
    while cur <= fin:
        stop = min(cur + pd.Timedelta(days=jours_max - 1), fin)
        data = api_get(
            "/v2/rates/candles", base=base, quote=quote,
            start_time=cur.strftime("%Y-%m-%d"), end_time=stop.strftime("%Y-%m-%d"),
            fields=list(fields), data_set=data_set,
        )
        morceaux.append(pd.DataFrame(data["quotes"]))
        cur = stop + pd.Timedelta(days=1)

    df = pd.concat(morceaux, ignore_index=True)
    # Les bornes peuvent se chevaucher d'un morceau à l'autre : on déduplique.
    df = df.drop_duplicates(subset=["base_currency", "quote_currency", "start_time"])
    for c in df.columns:
        if c.endswith(("_bid", "_ask", "_midpoint")):
            df[c] = pd.to_numeric(df[c])
    for c in ("start_time", "open_time", "close_time", "source_date"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True)
    return df.sort_values(["base_currency", "quote_currency", "start_time"]).reset_index(drop=True)


hist = historique("EUR", ["USD", "GBP"], "2025-01-01", "2025-12-31")
print(hist.shape)
hist.head()

In [ ]:
import matplotlib.pyplot as plt

eurusd = hist[(hist.base_currency == "EUR") & (hist.quote_currency == "USD")]
plt.figure(figsize=(10, 4))
plt.plot(eurusd["start_time"], eurusd["average_midpoint"], color="#0000FF", linewidth=1.5)
plt.title("EUR/USD, moyenne journalière (midpoint)")
plt.xlabel("Date (UTC)")
plt.ylabel("Taux")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Taux forward (selon abonnement)

`/v2/supported_forwards` liste les paires et échéances disponibles, `/v2/rates/forward` renvoie la courbe pour une paire. Une erreur 403 signifie en général que votre plan n'inclut pas les forwards.

In [ ]:
try:
    forwards_dispos = pd.DataFrame(api_get("/v2/supported_forwards", base=["EUR"], quote=["USD"])["supported_forwards"])
    print(forwards_dispos)
    fwd = pd.DataFrame(api_get("/v2/rates/forward", base="EUR", quote="USD")["quotes"])
    fwd
except RuntimeError as e:
    print("Forwards indisponibles :", e)

## 6. Exportation

In [ ]:
os.makedirs("exports", exist_ok=True)

hist.to_csv("exports/historique_eur.csv", index=False)
hist.to_json("exports/historique_eur.json", orient="records", date_format="iso", indent=2)

# Excel : les dates avec fuseau horaire doivent être retirées (UTC conservé dans le nom de colonne)
hist_xl = hist.copy()
for c in hist_xl.select_dtypes(include=["datetimetz"]).columns:
    hist_xl[c] = hist_xl[c].dt.tz_convert("UTC").dt.tz_localize(None)
    hist_xl = hist_xl.rename(columns={c: f"{c}_utc"})
hist_xl.to_excel("exports/historique_eur.xlsx", index=False)

print(os.listdir("exports"))